# Adapter Smoke Test

Directly load a trained continual SD-LoRA adapter on top of the DINO backbone, inspect its metadata, run one prediction, and optionally sanity-check a small folder of images.

This notebook does not use router inference.

In [ ]:
import sys\nfrom pathlib import Path\n\nROOT = Path.cwd().resolve()\nfor candidate in [ROOT, *ROOT.parents]:\n    if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():\n        ROOT = candidate\n        break\n\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom scripts.colab_repo_bootstrap import install_colab_requirements, mount_drive_if_available, resolve_repo_root, running_in_colab\n\nROOT = resolve_repo_root()\nif running_in_colab():\n    mount_drive_if_available()\n    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)\n\nprint(ROOT)

In [ ]:
import json\nfrom collections import Counter\nfrom pathlib import Path\n\nfrom IPython.display import display\n\ntry:\n    import pandas as pd\nexcept Exception:\n    pd = None\n\nfrom scripts.colab_adapter_smoke_test import load_adapter_summary, predict_image_folder, predict_single_image\nfrom scripts.colab_repo_bootstrap import running_in_colab\n\nCROP_NAME = 'tomato'\nADAPTER_DIR = None\n# Example explicit adapter directory:\n# ADAPTER_DIR = ROOT / 'outputs' / 'colab_notebook_training'\n# ADAPTER_DIR = ROOT / 'outputs' / 'colab_notebook_training' / 'continual_sd_lora_adapter'\n\nADAPTER_ROOT = None\n# Example deployed layout root:\n# ADAPTER_ROOT = ROOT / 'models' / 'adapters'\n\nIMAGE_PATH = None\nBATCH_IMAGE_DIR = None\nDEVICE = 'cuda'\n\nprint('ADAPTER_DIR takes priority over ADAPTER_ROOT when both are set.')

In [ ]:
summary = load_adapter_summary(\n    CROP_NAME,\n    adapter_dir=ADAPTER_DIR,\n    adapter_root=ADAPTER_ROOT,\n    config_env='colab',\n    device=DEVICE,\n)\n\nprint(json.dumps(summary, indent=2))

In [ ]:
if IMAGE_PATH is None:\n    if not running_in_colab():\n        raise ValueError('Set IMAGE_PATH when running outside Colab.')\n    from google.colab import files\n    uploaded = files.upload()\n    IMAGE_PATH = next(iter(uploaded.keys()))\n\nsingle_result = predict_single_image(\n    IMAGE_PATH,\n    CROP_NAME,\n    adapter_dir=ADAPTER_DIR,\n    adapter_root=ADAPTER_ROOT,\n    config_env='colab',\n    device=DEVICE,\n)\n\nprint(json.dumps(single_result, indent=2))

In [ ]:
if not BATCH_IMAGE_DIR:\n    print('Set BATCH_IMAGE_DIR to run the optional folder sanity check.')\nelse:\n    rows = predict_image_folder(\n        BATCH_IMAGE_DIR,\n        CROP_NAME,\n        adapter_dir=ADAPTER_DIR,\n        adapter_root=ADAPTER_ROOT,\n        config_env='colab',\n        device=DEVICE,\n    )\n    if pd is not None:\n        df = pd.DataFrame(rows)\n        display(df)\n    else:\n        df = None\n        print(rows)\n\n    predicted_counts = Counter(row['predicted_class'] for row in rows if row.get('predicted_class'))\n    ood_count = sum(1 for row in rows if row.get('is_ood') is True)\n    error_count = sum(1 for row in rows if row.get('error'))\n\n    print('predicted_class_counts:', dict(predicted_counts))\n    print('ood_count:', ood_count)\n    print('error_count:', error_count)